# Rhythm tuning & comparison

A focused version of  01_explore.ipynb

**Here we tune three parameters of interest :**
 tonic drive  
 background noise
 thalamic weight

Then run any of the five rhythmic patterns (with an optional
on-beat background) and compare them.

**We show the following:**

 stimulus/input
 population activity
 spike phase histograms
 E-E / I-I / E-I cross-correlograms 

**What is measured (plumbed from `experiments.measures`):**
- **syncopation** — LHL syncopation, offbeat count, WNBD (pattern-level)
- **synchrony** — PLV, mean vector strength, spike-count correlation (E and I)
- **signal to noise ratio** — single-neuron SNR (Czanner et al. 2015) on a subset of neurons


In [1]:
# Bookkeeping and imports
from pathlib import Path
import json
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Work when launched from either the repo root or the notebooks directory.
_here = Path.cwd().resolve()
_project = next((p for p in (_here, _here.parent) if (p / "src" / "model.py").exists()), None)
if _project is None:
    raise FileNotFoundError("Could not find auditory-entrainment/src/model.py")
if str(_project) not in sys.path:
    sys.path.insert(0, str(_project))

# Reload project modules so edits take effect on re-run.
import importlib
for _name in (
    "src.model",
    "src.stimuli",
    "experiments.measures.synchrony",
    "experiments.measures.syncopation",
    "experiments.measures.snr",
):
    if _name in sys.modules:
        importlib.reload(sys.modules[_name])

from src.model import Params, simulate
from src.stimuli import PATTERNS, PATTERN_NAMES, Rhythm, ONBEAT_BACKGROUND
from experiments.measures import synchrony, syncopation, snr

plt.rcParams.update({"figure.figsize": (12, 9), "axes.grid": True, "grid.alpha": 0.25})
print(f"Using project: {_project}")


Using project: C:\Users\local\Desktop\neural_entrainment_synchrony\auditory-entrainment


## Parameters of interest and stimulus

Set a profile and tune parameters directly, pick a rhythmic
condition, and press **Run simulation**.


In [ ]:
# Fixed network baseline (the scale and heterogeneity used for the sweep).
# duration and seed are exposed as widgets below; I_bias / bg_noise / w_thal
# are the three tuned parameters of interest.
FIXED_PARAMS = {
    "dt": 0.1,
    "v_dt": 1.0,
    "n_exc": 6400,
    "n_inh": 1600,
    "n_thal": 320,
    "synaptic_normalization": "sqrt",
    "n_exc_ref": 400,
    "n_inh_ref": 100,
    "n_thal_ref": 20,
    "C_m": 200.0,
    "tau_exc": 20.0,
    "tau_inh": 10.0,
    "E_l": -65.0,
    "E_e": 0.0,
    "E_i": -80.0,
    "v_thresh": -50.0,
    "v_reset": -65.0,
    "tau_ref_exc": 2.0,
    "tau_ref_inh": 1.0,
    "tau_e": 5.0,
    "tau_i": 10.0,
    "delay_ei": 1.5,
    "delay_ie": 1.5,
    "delay_ei_std": 0.5,
    "delay_ie_std": 0.5,
    "w_ee": 1.5,
    "w_ei": 0.85,
    "w_ie": 3.0,
    "w_ii": 4.0,
    "p_ee": 0.02,
    "p_ei": 0.4,
    "p_ie": 0.5,
    "p_ii": 0.5,
    "p_thal": 0.4,
    "tau_adapt": 150.0,
    "w_adapt": 100.0,
    "delta_adapt": 0.1,
    "weight_lognormal_sigma": 0.4,
}

field_layout = widgets.Layout(width="155px")
field_style = {"description_width": "72px"}

# Pattern / stimulus controls
_pattern_options = [(f"{pid}: {PATTERN_NAMES[pid]}  {PATTERNS[pid]}", pid) for pid in PATTERNS]
preset = widgets.Dropdown(options=_pattern_options, value=3, description="Pattern:")
step_duration = widgets.BoundedFloatText(value=0.25, min=0.05, max=1.0, step=0.05, description="step (s)", layout=field_layout, style=field_style)
rate_on = widgets.BoundedFloatText(value=40.0, min=5.0, max=100.0, step=5.0, description="on (Hz)", layout=field_layout, style=field_style)
show_background = widgets.Checkbox(value=True, description="on-beat background")
duration = widgets.BoundedIntText(value=20000, min=1000, max=60000, step=1000, description="dur (ms)", layout=field_layout, style=field_style)
seed = widgets.IntText(value=42, description="seed", layout=field_layout, style=field_style)

# Tunable parameters of interest
i_bias = widgets.BoundedFloatText(value=125.0, min=0.0, max=200.0, step=5.0, description="I_bias (pA)", layout=field_layout, style=field_style)
bg_noise = widgets.BoundedFloatText(value=3.0, min=0.0, max=10.0, step=0.5, description="bg_noise (mV)", layout=field_layout, style=field_style)
w_thal = widgets.BoundedFloatText(value=0.08, min=0.0, max=0.5, step=0.01, description="w_thal (nS)", layout=field_layout, style=field_style)

# SNR evaluation controls
snr_neurons = widgets.IntSlider(value=50, min=10, max=400, step=10, description="SNR neurons:")
snr_min_spikes = widgets.IntSlider(value=10, min=1, max=100, step=1, description="min spikes:")

# Parameter profiles: named sets saved from 01_explore (parameter_sets.json)
# plus the low/medium/high regimes found by the sweep (parameter_sweep.csv).
PARAMETER_SETS_FILE = _project / "notebooks" / "parameter_sets.json"

_SAVED_SETS = {}
if PARAMETER_SETS_FILE.exists():
    try:
        with PARAMETER_SETS_FILE.open(encoding="utf-8") as handle:
            _SAVED_SETS = json.load(handle)
    except (OSError, ValueError):
        pass

_SWEEP_PROFILES = {}
_sw_csv = _project / "results" / "parameter_sweep.csv"
if _sw_csv.exists():
    try:
        import pandas as pd
        df = pd.read_csv(_sw_csv)
        required = {"valid", "I_bias", "bg_noise", "w_thal", "plv", "vs", "xcorr", "mean_snr"}
        if required <= set(df.columns):
            from experiments.optimize_parameters import find_optimal_targets
            for _name, _row in find_optimal_targets(df).items():
                _SWEEP_PROFILES[f"sweep: {_name}"] = {
                    "I_bias": float(_row["I_bias"]),
                    "bg_noise": float(_row["bg_noise"]),
                    "w_thal": float(_row["w_thal"]),
                }
    except Exception:
        pass

profile_options = [("custom", "custom")]
profile_options += [(name, name) for name in _SAVED_SETS]
profile_options += [(name, name) for name in _SWEEP_PROFILES]
profile = widgets.Dropdown(
    options=profile_options or [("custom", "custom")],
    value="custom",
    description="Profile:",
)
apply_profile_button = widgets.Button(description="Apply profile", button_style="info", icon="wrench")
profile_status = widgets.Output()

run_button = widgets.Button(description="Run simulation", button_style="success", icon="play")
output = widgets.Output()


def _apply_saved_set(values):
    """Map a parameter_sets.json / sweep entry onto the tuning widgets.

    The explorer saves background noise under "noise" and the pattern id
    under "preset"; everything else shares its widget name. Missing keys
    are left untouched so partial/old sets still load.
    """
    if "I_bias" in values:
        i_bias.value = values["I_bias"]
    bg = values.get("bg_noise", values.get("noise"))
    if bg is not None:
        bg_noise.value = bg
    if "w_thal" in values:
        w_thal.value = values["w_thal"]
    if "preset" in values and values["preset"] in PATTERNS:
        preset.value = values["preset"]
    if "step_duration" in values:
        step_duration.value = values["step_duration"]
    if "rate_on" in values:
        rate_on.value = values["rate_on"]
    if "show_background" in values:
        show_background.value = bool(values["show_background"])
    if "duration" in values:
        duration.value = values["duration"]
    if "seed" in values:
        seed.value = values["seed"]


def apply_profile(_button=None):
    with profile_status:
        clear_output(wait=True)
        name = profile.value
        if name == "custom":
            print("Select a saved set or sweep profile to apply, or edit the parameters directly.")
            return
        values = _SAVED_SETS.get(name) or _SWEEP_PROFILES.get(name)
        if not values:
            print(f"No profile data for '{name}'.")
            return
        _apply_saved_set(values)
        print(f"Applied '{name}':")
        print(f"  I_bias={i_bias.value:g}, bg_noise={bg_noise.value:g}, w_thal={w_thal.value:g}")
        print(f"  pattern={preset.value} ({PATTERN_NAMES[preset.value]}), step={step_duration.value:g}s, "
              f"rate_on={rate_on.value:g}, background={show_background.value}, "
              f"duration={duration.value}ms, seed={seed.value}")


In [3]:
def _bin_trains(spike_times_s, spike_indices, n_units, duration_s, bin_s):
    n_units = int(n_units)
    n_bins = max(1, int(np.ceil(duration_s / bin_s)))
    spike_times_s = np.asarray(spike_times_s, dtype=float)
    spike_indices = np.asarray(spike_indices, dtype=np.int64)
    counts = np.zeros((n_units, n_bins), dtype=float)
    if spike_times_s.size and n_units > 0:
        keep = (spike_indices >= 0) & (spike_indices < n_units)
        times = spike_times_s[keep]
        idx = spike_indices[keep]
        bins = np.clip((times / bin_s).astype(np.int64), 0, n_bins - 1)
        flat = np.bincount(idx * n_bins + bins, minlength=n_units * n_bins)
        counts = flat.reshape(n_units, n_bins).astype(float)
    return counts - counts.mean(axis=1, keepdims=True)


def _average_xcorr(trains_a, trains_b=None, bin_ms=1.0, max_lag_ms=50.0):
    bin_s = bin_ms / 1000.0
    max_lag_bins = int(round(max_lag_ms / bin_ms))
    lags = np.arange(-max_lag_bins, max_lag_bins + 1) * bin_ms
    a = np.asarray(trains_a, dtype=float)
    b = None if trains_b is None else np.asarray(trains_b, dtype=float)
    if a.ndim != 2 or a.shape[0] < 1 or a.shape[1] == 0:
        return lags, np.zeros(lags.size)
    n_bins = a.shape[1]
    if b is None:
        if a.shape[0] < 2:
            return lags, np.zeros(lags.size)
        n_units = a.shape[0]
        agg = np.correlate(a.sum(axis=0), a.sum(axis=0), mode="full")
        acc = agg[n_bins - 1 - max_lag_bins: n_bins + max_lag_bins].copy()
        acc[max_lag_bins] -= float(np.einsum("ij,ij->", a, a))
        for k in range(1, max_lag_bins + 1):
            pos = float(np.einsum("ij,ij->", a[:, :n_bins - k], a[:, k:]))
            acc[max_lag_bins + k] -= pos
            acc[max_lag_bins - k] -= pos
        acc /= n_units * (n_units - 1)
        norm = n_bins * float(a.std(axis=1).mean()) ** 2
    else:
        if b.ndim != 2 or b.shape[0] < 1 or b.shape[1] != n_bins:
            return lags, np.zeros(lags.size)
        acc = np.correlate(a.sum(axis=0), b.sum(axis=0), mode="full") / (a.shape[0] * b.shape[0])
        acc = acc[n_bins - 1 - max_lag_bins: n_bins + max_lag_bins]
        norm = n_bins * float(a.std(axis=1).mean()) * float(b.std(axis=1).mean())
    if norm > 0:
        acc = acc / norm
    return lags, acc


def _phase_histogram(ax, spike_times_s, freq, color, title):
    phases = (np.asarray(spike_times_s, dtype=float) * freq) % 1.0
    angles = phases * 2.0 * np.pi
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(title, pad=16)
    if phases.size == 0:
        ax.bar([0.0], [0.0], width=0.1, color=color, alpha=0.0)
        ax.text(0.5, -0.2, "no spikes", transform=ax.transAxes, ha="center", va="top", fontsize=9)
        return
    ax.hist(angles, bins=24, color=color, alpha=0.75, density=True)
    z = np.mean(np.exp(1j * angles))
    mean_phase = (np.angle(z) / (2.0 * np.pi)) % 1.0
    ax.annotate("", xy=(np.angle(z), np.abs(z)), xytext=(0.0, 0.0), arrowprops=dict(arrowstyle="->", color="black", lw=2.0))
    ax.text(0.5, -0.2, f"VS={np.abs(z):.3f} | mean phase={mean_phase:.2f} cyc", transform=ax.transAxes, ha="center", va="top", fontsize=9)


In [4]:
def _report_measures(result, rhythm, freq):
    duration_s = result.params.duration / 1000.0
    fs = 1.0 / float(np.mean(np.diff(result.rate_t)))
    exc_mask = result.spike_indices < result.n_exc
    exc_times = result.spike_times[exc_mask]
    exc_idx = result.spike_indices[exc_mask]
    inh_times = result.spike_times[~exc_mask]
    inh_idx = result.spike_indices[~exc_mask] - result.n_exc

    # Pattern syncopation (foreground rhythm)
    lhl = syncopation.lhl_syncopation(rhythm.pattern)
    offbeat = syncopation.offbeat_count(rhythm.pattern)
    wnbd = syncopation.wnbd(rhythm.pattern)

    # Network synchrony
    plv_e = synchrony.plv(result.rate_exc, fs, freq)
    plv_i = synchrony.plv(result.rate_inh, fs, freq)
    vs_e = synchrony.mean_vector_strength(exc_times, exc_idx, result.n_exc, freq)
    vs_i = synchrony.mean_vector_strength(inh_times, inh_idx, result.n_inh, freq)
    corr_e = synchrony.spike_pairwise_corr(exc_times, exc_idx, result.n_exc, duration_s)
    corr_i = synchrony.spike_pairwise_corr(inh_times, inh_idx, result.n_inh, duration_s)

    # Single-neuron SNR on a subset of excitatory neurons
    candidates = np.unique(exc_idx)
    rng = np.random.default_rng(result.params.seed + 1000)
    n_sample = min(int(snr_neurons.value), candidates.size)
    neuron_ids = rng.choice(candidates, size=n_sample, replace=False) if n_sample else np.asarray([], dtype=np.int64)
    fitted = snr.population_snr(
        exc_times, exc_idx, result.n_exc, duration_s, freq,
        bin_ms=5.0, n_lags=20, min_spikes=int(snr_min_spikes.value), neuron_ids=neuron_ids,
    )
    snr_vals = np.asarray([r["snr_db"] for r in fitted if np.isfinite(r["snr_db"])], dtype=float)

    print(f"Pattern: {rhythm.name} {list(rhythm.pattern)} | fundamental {freq:.2f} Hz")
    print(f"Syncopation: LHL={lhl} | offbeat={offbeat} | WNBD={wnbd:.3f}")
    print(f"Mean rates: E {float(result.rate_exc.mean()):.2f} Hz | I {float(result.rate_inh.mean()):.2f} Hz")
    print(f"PLV:        E {plv_e:.3f} | I {plv_i:.3f}")
    print(f"VS (mean):  E {vs_e:.3f} | I {vs_i:.3f}")
    print(f"Spike corr: E {corr_e:.3f} | I {corr_i:.3f}")
    if snr_vals.size:
        print(f"SNR (E, {snr_vals.size} neurons): mean={snr_vals.mean():.2f} dB | median={np.median(snr_vals):.2f} dB | max={snr_vals.max():.2f} dB")
    else:
        print("SNR (E): no finite values (increase duration or lower min spikes)")
    return {
        "lhl": lhl, "offbeat": offbeat, "wnbd": wnbd,
        "plv_e": plv_e, "plv_i": plv_i, "vs_e": vs_e, "vs_i": vs_i,
        "corr_e": corr_e, "corr_i": corr_i,
        "snr_mean": float(snr_vals.mean()) if snr_vals.size else float("nan"),
        "snr_median": float(np.median(snr_vals)) if snr_vals.size else float("nan"),
        "snr_max": float(snr_vals.max()) if snr_vals.size else float("nan"),
    }


def _plot_result(result, rhythm, background=None):
    duration_s = result.params.duration / 1000.0
    freq = rhythm.pulse_frequency
    t_stim = np.linspace(0.0, duration_s, 2000, endpoint=False)
    stimulus_rate = rhythm.rate(t_stim)
    n_pattern_steps = int(np.ceil(duration_s / rhythm.step_duration))
    pattern_values = np.tile(rhythm.pattern, n_pattern_steps + 1)[:n_pattern_steps + 1]
    pattern_times = np.arange(pattern_values.size) * rhythm.step_duration * 1000
    t_rate = result.rate_t * 1000
    exc_mask = result.spike_indices < result.n_exc
    exc_spike_s = result.spike_times[exc_mask]
    inh_spike_s = result.spike_times[~exc_mask]
    exc_idx = result.spike_indices[exc_mask]
    inh_idx = result.spike_indices[~exc_mask] - result.n_exc

    fig = plt.figure(figsize=(14, 14))
    gs = fig.add_gridspec(4, 1, height_ratios=[1.0, 1.0, 1.0, 1.0], hspace=0.7)

    # Row 0: input
    ax = fig.add_subplot(gs[0])
    ax.plot(t_stim * 1000, stimulus_rate, color="tab:orange", lw=1.5, label="selected stimulus")
    if background is not None:
        ax.plot(t_stim * 1000, background.rate(t_stim), color="tab:green", lw=1.3, linestyle="--", label="on-beat background")
    ax.step(pattern_times, pattern_values, where="post", color="black", alpha=0.7, label="onset pattern")
    ax.set_ylabel("Input (Hz) / onset")
    ax.set_title(f"Stimulus: {rhythm.name} | cycle {rhythm.cycle_duration:.2f} s | fundamental {freq:.2f} Hz")
    ax.set_xlim(0.0, duration_s * 1000.0)
    ax.legend(loc="upper right")

    # Row 1: population activity
    ax = fig.add_subplot(gs[1])
    ax.plot(t_rate, result.rate_exc, color="tab:blue", label="excitatory rate")
    ax.plot(t_rate, result.rate_inh, color="tab:red", label="inhibitory rate")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Rate (Hz)")
    ax.set_title("Population activity")
    ax.set_xlim(0.0, duration_s * 1000.0)
    ax.legend(loc="upper right")

    # Row 2: spike phase
    sub_polar = gs[2].subgridspec(1, 2, wspace=0.35)
    _phase_histogram(fig.add_subplot(sub_polar[0], projection="polar"), exc_spike_s, freq, "tab:blue", "Excitatory spike phase")
    _phase_histogram(fig.add_subplot(sub_polar[1], projection="polar"), inh_spike_s, freq, "tab:red", "Inhibitory spike phase")

    # Row 3: cross-correlogram
    bin_ms, max_lag_ms = 1.0, 50.0
    bin_s = bin_ms / 1000.0
    trains_e = _bin_trains(exc_spike_s, exc_idx, result.n_exc, duration_s, bin_s)
    trains_i = _bin_trains(inh_spike_s, inh_idx, result.n_inh, duration_s, bin_s)
    lags, xcorr_ee = _average_xcorr(trains_e, bin_ms=bin_ms, max_lag_ms=max_lag_ms)
    _, xcorr_ii = _average_xcorr(trains_i, bin_ms=bin_ms, max_lag_ms=max_lag_ms)
    _, xcorr_ei = _average_xcorr(trains_e, trains_i, bin_ms=bin_ms, max_lag_ms=max_lag_ms)
    sub_x = gs[3].subgridspec(1, 3, wspace=0.3)
    for pos, values, title, color in (
        (0, xcorr_ee, "E-E", "tab:blue"),
        (1, xcorr_ii, "I-I", "tab:red"),
        (2, xcorr_ei, "E-I", "tab:purple"),
    ):
        axis = fig.add_subplot(sub_x[pos])
        axis.axhline(0.0, color="grey", lw=0.6)
        axis.bar(lags, values, width=bin_ms * 0.9, color=color, alpha=0.85)
        axis.set_xlabel("Lag (ms)")
        axis.set_ylabel("Correlation")
        axis.set_title(f"{title} cross-correlogram")

    plt.show(fig)
    plt.close(fig)


def run_simulation(_button=None):
    with output:
        clear_output(wait=True)
        try:
            pattern = tuple(PATTERNS[preset.value])
            label = PATTERN_NAMES[preset.value]
            rhythm = Rhythm(pattern, step_duration=step_duration.value, rate_on=rate_on.value, jitter_ms=0.0, name=label)
            background = (
                Rhythm(ONBEAT_BACKGROUND, step_duration=step_duration.value, rate_on=rate_on.value, jitter_ms=0.0, name="on-beat background")
                if show_background.value
                else None
            )
            params = Params(
                **FIXED_PARAMS,
                duration=float(duration.value),
                seed=int(seed.value),
                I_bias=float(i_bias.value),
                bg_noise=float(bg_noise.value),
                w_thal=float(w_thal.value),
            )
            t0 = time.perf_counter()
            print("Running Brian2 simulation...")
            result = simulate(params, stimulus=rhythm, background=background)
            t1 = time.perf_counter()
            print(f"  simulation: {t1 - t0:.1f} s")
            freq = rhythm.pulse_frequency
            _plot_result(result, rhythm, background=background)
            t2 = time.perf_counter()
            print(f"  plot:       {t2 - t1:.1f} s")
            _report_measures(result, rhythm, freq)
            print(f"  total:      {time.perf_counter() - t0:.1f} s")
        except Exception as exc:
            print(f"Could not run simulation: {type(exc).__name__}: {exc}")


# Bind handlers idempotently. A per-widget flag (not a global) means each
# button keeps exactly one click handler no matter how many times this cell
# or the widgets cell above is re-run.
for _button, _handler in (
    (run_button, run_simulation),
    (apply_profile_button, apply_profile),
):
    if not getattr(_button, "_bound_once", False):
        _button.on_click(_handler)
        _button._bound_once = True

controls = widgets.VBox([
    widgets.HBox([profile, apply_profile_button, profile_status]),
    widgets.HBox([i_bias, bg_noise, w_thal]),
    widgets.HBox([preset, step_duration, rate_on, show_background]),
    widgets.HBox([duration, seed, snr_neurons, snr_min_spikes]),
    run_button,
])

display(controls, output)


Output()